In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson

# -----------------------------
# SETTINGS
# -----------------------------
ROLLING_GAMES = 15
SMOOTHING_K = 5
MAX_GOALS = 10
EDGE_THRESHOLD = 0.02
EV_THRESHOLD = 0.05 # Define EV_THRESHOLD

OPTIMIZE = True

RHO = -0.1
DECAY_RATE = 0.02

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("EPLmatches2.csv")

df['HomeTeam'] = df['HomeTeam'].str.strip()
df['AwayTeam'] = df['AwayTeam'].str.strip()

df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
df = df.dropna(subset=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# -----------------------------
# WEIGHTED AVERAGE HELPER
# -----------------------------
def wavg(values, weights):
    if len(values) == 0 or np.sum(weights) == 0:
        return 0
    vals = np.array(values)
    wts = np.array(weights)
    return np.sum(vals * wts) / np.sum(wts)

# -----------------------------
# TEAM STATS (HOME/AWAY SPLIT)
# -----------------------------
def get_team_stats(team, df, current_date, league_home, league_away):

    games = df[((df['HomeTeam'] == team) | (df['AwayTeam'] == team)) &
               (df['Date'] < current_date)]

    games = games.tail(ROLLING_GAMES)

    if len(games) == 0:
        return {
            'home_attack': league_home,
            'home_defense': league_away,
            'away_attack': league_away,
            'away_defense': league_home
        }

    ha, ha_w = [], []
    hd, hd_w = [], []
    aa, aa_w = [], []
    ad, ad_w = [], []

    for i, (_, row) in enumerate(games.iterrows()):
        # Calculate decay weight, giving more weight to recent games
        w = np.exp(-DECAY_RATE * (len(games) - 1 - i))

        if row['HomeTeam'] == team:
            ha.append(row['HomeGoals'])
            ha_w.append(w)
            hd.append(row['AwayGoals']) # Goals conceded by home team
            hd_w.append(w)
        else: # AwayTeam == team
            aa.append(row['AwayGoals'])
            aa_w.append(w)
            ad.append(row['HomeGoals']) # Goals conceded by away team
            ad_w.append(w)

    n = len(games)

    home_attack = (wavg(ha, ha_w)*n + SMOOTHING_K*league_home)/(n+SMOOTHING_K)
    home_defense = (wavg(hd, hd_w)*n + SMOOTHING_K*league_away)/(n+SMOOTHING_K)
    away_attack = (wavg(aa, aa_w)*n + SMOOTHING_K*league_away)/(n+SMOOTHING_K)
    away_defense = (wavg(ad, ad_w)*n + SMOOTHING_K*league_home)/(n+SMOOTHING_K)

    return {
        'home_attack': home_attack,
        'home_defense': home_defense,
        'away_attack': away_attack,
        'away_defense': away_defense
    }

# -----------------------------
# DIXON-COLES
# -----------------------------
def dixon_coles(h, a, lam_h, lam_a, rho):

    if h == 0 and a == 0:
        return 1 - (lam_h * lam_a * rho)
    elif h == 1 and a == 1:
        return 1 + rho
    elif (h == 1 and a == 0) or (h == 0 and a == 1):
        return 1 - rho
    return 1

# -----------------------------
# EXPECTED GOALS
# -----------------------------
def expected_goals(ht, at, past, current_date):

    league_home = past['HomeGoals'].mean()
    league_away = past['AwayGoals'].mean()

    hs = get_team_stats(ht, past, current_date, league_home, league_away)
    aws = get_team_stats(at, past, current_date, league_home, league_away)

    lam_h = hs['home_attack'] * aws['away_defense']
    lam_a = aws['away_attack'] * hs['home_defense']

    return lam_h, lam_a

# -----------------------------
# MATCH PROBABILITIES
# -----------------------------
def match_probabilities(lam_h, lam_a):

    home = away = raw_draw_p = 0
    raw_total_p = 0

    for h in range(MAX_GOALS + 1):
        for a in range(MAX_GOALS + 1):

            p = poisson.pmf(h, lam_h) * poisson.pmf(a, lam_a)
            p *= dixon_coles(h, a, lam_h, lam_a, RHO)

            raw_total_p += p

            if h > a:
                home += p
            elif h == a:
                raw_draw_p += p
            else:
                away += p

    # Apply the boosting factor to the raw draw probability
    adjusted_draw_p = raw_draw_p * 1.1

    # Recalculate the total probability including the adjusted draw for normalization
    adjusted_total_p = raw_total_p - raw_draw_p + adjusted_draw_p

    if adjusted_total_p == 0: # Avoid division by zero
        return 0, 0, 0

    # Normalize the probabilities
    home_p = home / adjusted_total_p
    draw_p = adjusted_draw_p / adjusted_total_p
    away_p = away / adjusted_total_p

    return home_p, draw_p, away_p

# -----------------------------
# PARAMETER OPTIMIZATION (FAST)
# -----------------------------
def optimize_parameters(df, sample_size=300):

    global RHO, DECAY_RATE

    best_score = -np.inf
    best_params = (RHO, DECAY_RATE)

    sample = df.tail(sample_size)

    for rho in [-0.1, -0.05]:
        for decay in [0.02, 0.03]:

            RHO = rho
            DECAY_RATE = decay

            log_likelihood = 0
            count = 0

            for i in range(len(sample)):

                row = sample.iloc[i]
                idx = df.index.get_loc(row.name)
                past = df.iloc[:idx]

                if len(past) < 50:
                    continue

                lam_h, lam_a = expected_goals(
                    row['HomeTeam'], row['AwayTeam'], past, row['Date']
                )

                h, a = row['HomeGoals'], row['AwayGoals']

                p = poisson.pmf(h, lam_h) * poisson.pmf(a, lam_a)
                p *= dixon_coles(h, a, lam_h, lam_a, RHO)

                if p <= 0:
                  continue

                if p > 0:
                    log_likelihood += np.log(p)
                    count += 1

            if count > 0 and log_likelihood > best_score:
                best_score = log_likelihood
                best_params = (rho, decay)

    RHO, DECAY_RATE = best_params

    print(f"\nOptimized RHO: {RHO}")
    print(f"Optimized DECAY: {DECAY_RATE}")

# -----------------------------
# VALUE BETTING
# -----------------------------
def value_bets(home_p, draw_p, away_p, home_odds, draw_odds, away_odds):

    imp_h = 1 / home_odds
    imp_d = 1 / draw_odds
    imp_a = 1 / away_odds

    # Normalize implied probabilities to remove the bookmaker's overround
    total_implied_prob = imp_h + imp_d + imp_a
    imp_h_norm = imp_h / total_implied_prob
    imp_d_norm = imp_d / total_implied_prob
    imp_a_norm = imp_a / total_implied_prob

    # Calculate "edge" for Home and Draw
    edge_h = home_p - imp_h_norm
    edge_d = draw_p - imp_d_norm

    # The original code adjusts away_p. Let's use an adjusted version for calculations.
    adjusted_away_p = away_p ** 1.1

    # Calculate "edge" for Away using the adjusted probability
    edge_a = adjusted_away_p - imp_a_norm

    # Calculate Expected Value (EV) for all outcomes
    ev_h = home_p * home_odds - 1 # Calculate ev_h
    ev_d = draw_p * draw_odds - 1 # Calculate ev_d
    ev_a = adjusted_away_p * away_odds - 1

    MIN_PROB_HOME = 0.10
    MIN_PROB_DRAW = 0.12
    MIN_PROB_AWAY = 0.15

    print("\n--- EDGES ---")
    print(f"Home: {edge_h:.3f}")
    print(f"Draw: {edge_d:.3f}")
    print(f"Away (Edge): {edge_a:.3f}")
    print(f"Away (EV): {ev_a:.3f}")
    print(f"Home (EV): {ev_h:.3f}") # Print ev_h for debugging/insight
    print(f"Draw (EV): {ev_d:.3f}") # Print ev_d for debugging/insight

    print("\n--- BETS ---")
    bet_found = False
    if home_p > MIN_PROB_HOME and ev_h > EV_THRESHOLD:
      print("BET: Home")
      bet_found = True

    if draw_p > MIN_PROB_DRAW and ev_d > EV_THRESHOLD:
      print("BET: Draw")
      bet_found = True
    if away_p > MIN_PROB_AWAY and ev_a > EV_THRESHOLD:
        print("BET: Away")
        bet_found = True

    if not bet_found:
        print("No value bets")

# -----------------------------
# BACKTEST
# -----------------------------
def backtest(df, sample_size=500):

    correct = total = 0

    sample = df.tail(sample_size)

    for i in range(len(sample)):

        row = sample.iloc[i]
        idx = df.index.get_loc(row.name)
        past = df.iloc[:idx]

        if len(past) < 50:
            continue

        lam_h, lam_a = expected_goals(
            row['HomeTeam'], row['AwayTeam'], past, row['Date']
        )

        home_p, draw_p, away_p = match_probabilities(lam_h, lam_a)

        probs = {"H": home_p, "D": draw_p, "A": away_p}
        pred = max(probs, key=probs.get)

        actual = (
            "H" if row['HomeGoals'] > row['AwayGoals']
            else "A" if row['HomeGoals'] < row['AwayGoals']
            else "D"
        )

        if pred == actual:
            correct += 1

        total += 1

    print(f"\nBacktest Accuracy: {correct/total:.3f} ({total} games)")

# -----------------------------
# RUN
# -----------------------------
if OPTIMIZE:
    print("Optimizing parameters...")
    optimize_parameters(df)

home = input("\nHome team: ")
away = input("Away team: ")

lam_h, lam_a = expected_goals(
    home, away,
    df.iloc[:-1],
    df.iloc[-1]['Date']
)

home_p, draw_p, away_p = match_probabilities(lam_h, lam_a)

print("\n--- PROBABILITIES ---")
print(home, round(home_p, 3))
print("Draw", round(draw_p, 3))
print(away, round(away_p, 3))

# -----------------------------
# ODDS INPUT
# -----------------------------
print("\nEnter American odds:")

def american_to_decimal(odds):
    if odds > 0:
        return 1 + (odds / 100)
    else:
        return 1 + (100 / abs(odds))

ho = american_to_decimal(float(input("Home odds: ")))
do = american_to_decimal(float(input("Draw odds: ")))
ao = american_to_decimal(float(input("Away odds: ")))




value_bets(home_p, draw_p, away_p, ho, do, ao)

backtest(df, 500)

Optimizing parameters...

Optimized RHO: -0.1
Optimized DECAY: 0.02

Home team: Tottenham
Away team: Nott'm Forest

--- PROBABILITIES ---
Tottenham 0.425
Draw 0.207
Nott'm Forest 0.368

Enter American odds:
Home odds: 130
Draw odds: 240
Away odds: 210

--- EDGES ---
Home: 0.012
Draw: -0.073
Away (Edge): 0.026
Away (EV): 0.031
Home (EV): -0.022
Draw (EV): -0.296

--- BETS ---
No value bets

Backtest Accuracy: 0.480 (500 games)
